pai_nosso_refatorado.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/github/alanabdmorais/pai_nosso_refatorado/blob/main/pai_nosso_refatorado.ipynb

# 🎬 Pipeline Multilíngue — Orações com Legendas Morfológicas

**Estratégia de classificação intermediária:**
O Groq gera as classificações → você revisa com uma IA → pipeline continua com os JSONs corrigidos.

---

### Fluxo completo

| # | Fase | O que faz | Ação |
|---|------|-----------|------|
| 0 | Setup | Instala deps, monta Drive, importa módulos | Automático |
| Init | — | Cria config, groq e pipeline | Automático |
| B0 | YouTube | Baixa legendas de referência (opcional) | Manual |
| 1 | Áudio | Edge TTS → .wav no Drive | Automático |
| 2 | Whisper | Transcrição → SRT bruto | Automático |
| 3 | Correção PT | Groq corrige o SRT | Automático |
| 4 | Traduções | Groq gera EN/ES/FR | Automático |
| **5A** | **Classificação** | **Groq classifica + gera pacote de revisão → pausa** | **Automático → pausa** |
| **5B** | **Revisão** | **Você corrige JSONs com IA externa** | **👤 Manual** |
| **5C** | **Recarregar** | **Pipeline carrega JSONs corrigidos** | **Automático** |
| 6 | Clipes | Baixa e corta vídeos do Pixabay | Automático |
| 7 | Vídeo base | Crédito + narração + trilha | Automático |
| 8 | Legendas ASS | Queima legendas coloridas no vídeo | Automático |

> **Retomar de uma fase:** `pipeline.run(from_phase='nome_da_fase')`

In [1]:
# ╔══════════════════════════════════════════════════╗
# ║  CÉLULA 0 — Setup (rode uma vez por sessão)    ║
# ╚══════════════════════════════════════════════════╝

# Sistema
!apt-get -qq -y install ffmpeg espeak-ng > /dev/null 2>&1
print('✅ ffmpeg + espeak-ng')

# Python
!pip install -q edge-tts openai-whisper openai pandas gdown yt-dlp nest_asyncio
print('✅ pacotes Python')

# Drive
from google.colab import drive, userdata
drive.mount('/content/drive')
print('✅ Drive montado')

# Copiar módulos do Drive para /content/pipeline
import shutil, os, sys, logging

PASTA_MODULOS = '/content/drive/MyDrive/pipeline/modulos'  # <-- ajuste se necessário
DESTINO       = '/content/pipeline'

if os.path.exists(PASTA_MODULOS):
    shutil.copytree(PASTA_MODULOS, DESTINO, dirs_exist_ok=True)
    print(f'✅ Módulos copiados: {PASTA_MODULOS} → {DESTINO}')
else:
    print(f'⚠️  Pasta não encontrada: {PASTA_MODULOS}')
    print('   Ajuste PASTA_MODULOS acima.')

if DESTINO not in sys.path:
    sys.path.insert(0, DESTINO)

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(name)-22s  %(levelname)s  %(message)s',
    datefmt='%H:%M:%S',
)

os.chdir('/content')
print('✅ Pronto.')

✅ ffmpeg + espeak-ng
✅ pacotes Python
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive montado
✅ Módulos copiados: /content/drive/MyDrive/pipeline/modulos → /content/pipeline
✅ Pronto.


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  INICIALIZAÇÃO — rode após o Setup                              ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
import nest_asyncio
nest_asyncio.apply()

from pathlib import Path
from config import PipelineConfig
from groq_client import GroqClient
from video_pipeline import VideoPipeline
from checkpoint import Checkpoint
from google.colab import userdata

# ── Ajuste aqui para trocar a oração ─────────────────────────────────────────
config = PipelineConfig(
    NOME_ORACAO  = 'pai_nosso',
    # Para outra oração:
    # NOME_ORACAO  = 'ave_maria',
    # TEXTO_ORACAO = 'Ave Maria, cheia de graça...',
    # VOZ_EDGE     = 'pt-BR-FranciscaNeural',
)

groq     = GroqClient(api_key=userdata.get('GROQ_KEY'), nome_oracao=config.NOME_ORACAO)
pipeline = VideoPipeline(config, groq)

cp = Checkpoint()
print(config.resumo())
print()
print(cp.resumo())
print(f'\n▶  Próxima fase: {cp.proxima_fase_pendente() or "(tudo concluído)"}')

Oração:        pai_nosso
Áudio:         pai_nosso_audio.wav
SRT mestre:    pai_nosso_pt.srt
Vídeo final:   pai_nosso_final.mp4
Idiomas:       pt, en, es, fr
Voz Edge TTS:  pt-BR-AntonioNeural
Modelo Groq:   llama-3.3-70b-versatile
Duração clipe: 5s
Correcoes:     /content/drive/MyDrive/pipeline/correcoes/pai_nosso
Brutos:        /content/drive/MyDrive/pipeline/brutos/pai_nosso

── Checkpoint ──────────────────────────────
  audio_gerado                 ⬜ pendente
  srt_pt_bruto                 ⬜ pendente
  srt_pt_corrigido             ⬜ pendente
  srt_traduzidos               ⬜ pendente
  classificacoes_feitas        ⬜ pendente
  clipes_cortados              ⬜ pendente
  video_base_criado            ⬜ pendente
  legendas_queimadas           ⬜ pendente
────────────────────────────────────────────

▶  Próxima fase: audio_gerado


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 VERIFICAÇÃO DA ESTRUTURA DO DRIVE                           ║
# ║  (execute após a Inicialização para conferir o Drive)           ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path

print("═" * 60)
print("📁 VERIFICANDO ESTRUTURA DO DRIVE")
print("═" * 60)

pasta_correcoes = Path(f'/content/drive/MyDrive/pipeline/correcoes/{config.NOME_ORACAO}')
pasta_brutos    = Path(f'/content/drive/MyDrive/pipeline/brutos/{config.NOME_ORACAO}')

for rotulo, pasta in [("correcoes", pasta_correcoes), ("brutos", pasta_brutos)]:
    print(f"\n📁 {pasta}")
    if pasta.exists():
        arquivos = list(pasta.iterdir())
        if arquivos:
            for f in sorted(arquivos):
                print(f"   ✅ {f.name}  ({f.stat().st_size/1024:.1f} KB)")
        else:
            print("   (pasta vazia)")
    else:
        print("   ⚠️  Ainda não criada (será criada automaticamente na FASE 5A)")

print("\n" + "═" * 60)
print("💡 ESTRUTURA ESPERADA APÓS A FASE 5A:")
print(f"   MyDrive/pipeline/brutos/{config.NOME_ORACAO}/")
print(f"   └── classificacao_{config.NOME_ORACAO}_pt/en/es/fr.json")
print(f"   MyDrive/pipeline/correcoes/{config.NOME_ORACAO}/")
print(f"   ├── prompt_revisao.md")
print(f"   ├── relatorio_classificacoes.csv")
print(f"   └── classificacao_{config.NOME_ORACAO}_pt/en/es/fr.json")
print("═" * 60)

════════════════════════════════════════════════════════════
📁 VERIFICANDO ESTRUTURA DO DRIVE
════════════════════════════════════════════════════════════

📁 /content/drive/MyDrive/pipeline/correcoes/pai_nosso
   ✅ prompt_revisao.md  (4.3 KB)
   ✅ relatorio_classificacoes.csv  (16.3 KB)

📁 /content/drive/MyDrive/pipeline/brutos/pai_nosso
   (pasta vazia)

════════════════════════════════════════════════════════════
💡 ESTRUTURA ESPERADA APÓS A FASE 5A:
   MyDrive/pipeline/brutos/pai_nosso/
   └── classificacao_pai_nosso_pt/en/es/fr.json
   MyDrive/pipeline/correcoes/pai_nosso/
   ├── prompt_revisao.md
   ├── relatorio_classificacoes.csv
   └── classificacao_pai_nosso_pt/en/es/fr.json
════════════════════════════════════════════════════════════


In [36]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🧹 LIMPEZA SELETIVA - Compatível GitHub (sem widgets)          ║
# ║  Escolha o número das opções que quer limpar                    ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import shutil
import sys
import os

def limpeza_seletiva():
    print("═" * 65)
    print("🧹 LIMPEZA SELETIVA")
    print("═" * 65)
    print("\nEscolha o que limpar (digite os números separados por vírgula)")
    print("Exemplo: 1,3,5")
    print("═" * 65)
    print("  1 - 🎵 Áudios (.wav, .mp3)")
    print("  2 - 📝 Legendas (.srt, .ass) [mantém yt_ref]")
    print("  3 - 🎬 Vídeos gerados (base, final, clipes)")
    print("  4 - 📊 JSONs BRUTOS (pasta brutos/ no Drive)")
    print("  5 - ✅ JSONs CORRIGIDOS (pasta correcoes/ no Drive)")
    print("  6 - 📌 Checkpoint (checkpoint.json)")
    print("  7 - 📁 Pastas temporárias (clipes_cortados, temp_raw)")
    print("  8 - 📦 Cache Python (módulos carregados)")
    print("  9 - 🔥 LIMPAR TUDO (exceto Drive)")
    print(" 10 - 🔥🔥 LIMPAR TUDO + DRIVE (TUDO)")
    print("  0 - Sair")
    print("═" * 65)

    opcoes = input("\nDigite sua escolha: ").strip()

    if opcoes == '0':
        print("Saindo...")
        return

    opcoes_lista = [int(x.strip()) for x in opcoes.split(',')]
    cont = 0

    # 1. Áudios
    if 1 in opcoes_lista or 9 in opcoes_lista or 10 in opcoes_lista:
        print("\n🎵 Removendo áudios...")
        for f in Path('.').glob('*_audio.wav'):
            f.unlink()
            cont += 1
            print(f"   🗑️ {f.name}")
        for f in Path('.').glob('*.mp3'):
            if 'musica' not in f.name:
                f.unlink()
                cont += 1
                print(f"   🗑️ {f.name}")

    # 2. Legendas
    if 2 in opcoes_lista or 9 in opcoes_lista or 10 in opcoes_lista:
        print("\n📝 Removendo legendas...")
        for f in Path('.').glob('*.srt'):
            if 'yt_ref' not in f.name:
                f.unlink()
                cont += 1
                print(f"   🗑️ {f.name}")
        for f in Path('.').glob('*.ass'):
            f.unlink()
            cont += 1
            print(f"   🗑️ {f.name}")

    # 3. Vídeos
    if 3 in opcoes_lista or 9 in opcoes_lista or 10 in opcoes_lista:
        print("\n🎬 Removendo vídeos...")
        patterns = ['*_base.mp4', '*_final.mp4', 'clipe_*.mp4', 'temp_*.mp4', 'video_*.mp4', 'raw_*.mp4']
        for pattern in patterns:
            for f in Path('.').glob(pattern):
                f.unlink()
                cont += 1
                print(f"   🗑️ {f.name}")

    # 4. JSONs BRUTOS (pasta brutos/ no Drive)
    if 4 in opcoes_lista or 10 in opcoes_lista:
        print("\n📊 Removendo JSONs BRUTOS (pasta brutos/)...")
        pasta_brutos = Path('/content/drive/MyDrive/pipeline/brutos')
        if pasta_brutos.exists():
            for oracao_dir in pasta_brutos.iterdir():
                if oracao_dir.is_dir():
                    for f in oracao_dir.glob('classificacao_*.json'):
                        f.unlink()
                        cont += 1
                        print(f"   🗑️ Drive/brutos/{oracao_dir.name}/{f.name}")
        else:
            print("   ⚠️ Pasta brutos/ não encontrada")

    # 5. JSONs CORRIGIDOS (pasta correcoes/ no Drive)
    if 5 in opcoes_lista or 10 in opcoes_lista:
        print("\n✅ Removendo JSONs CORRIGIDOS (pasta correcoes/)...")
        pasta_correcoes = Path('/content/drive/MyDrive/pipeline/correcoes')
        if pasta_correcoes.exists():
            for oracao_dir in pasta_correcoes.iterdir():
                if oracao_dir.is_dir():
                    for f in oracao_dir.glob('classificacao_*.json'):
                        f.unlink()
                        cont += 1
                        print(f"   🗑️ Drive/correcoes/{oracao_dir.name}/{f.name}")
        else:
            print("   ⚠️ Pasta correcoes/ não encontrada")

    # 6. Checkpoint
    if 6 in opcoes_lista or 9 in opcoes_lista or 10 in opcoes_lista:
        print("\n📌 Removendo checkpoint...")
        cp = Path('checkpoint.json')
        if cp.exists():
            cp.unlink()
            cont += 1
            print("   🗑️ checkpoint.json")

    # 7. Pastas temporárias
    if 7 in opcoes_lista or 9 in opcoes_lista or 10 in opcoes_lista:
        print("\n📁 Removendo pastas temporárias...")
        pastas = ['clipes_cortados', 'clipes_prontos', 'temp_raw', '__pycache__']
        for pasta in pastas:
            p = Path(pasta)
            if p.exists():
                shutil.rmtree(p)
                cont += 1
                print(f"   🗑️ {pasta}/")

    # 8. Cache Python
    if 8 in opcoes_lista or 9 in opcoes_lista or 10 in opcoes_lista:
        print("\n📦 Limpando cache Python...")
        modulos = ['groq_client', 'video_pipeline', 'config', 'classification',
                   'checkpoint', 'drive_utils', 'ffmpeg_utils', 'srt_utils',
                   'models', 'constants', 'pipeline']
        for m in modulos:
            if m in sys.modules:
                del sys.modules[m]
                cont += 1
                print(f"   🗑️ {m}")

    print("\n" + "═" * 65)
    print(f"✅ LIMPEZA CONCLUÍDA! {cont} item(ns) removido(s)")
    print("═" * 65)

# Executar
limpeza_seletiva()

═════════════════════════════════════════════════════════════════
🧹 LIMPEZA SELETIVA
═════════════════════════════════════════════════════════════════

Escolha o que limpar (digite os números separados por vírgula)
Exemplo: 1,3,5
═════════════════════════════════════════════════════════════════
  1 - 🎵 Áudios (.wav, .mp3)
  2 - 📝 Legendas (.srt, .ass) [mantém yt_ref]
  3 - 🎬 Vídeos gerados (base, final, clipes)
  4 - 📊 JSONs BRUTOS (pasta brutos/ no Drive)
  5 - ✅ JSONs CORRIGIDOS (pasta correcoes/ no Drive)
  6 - 📌 Checkpoint (checkpoint.json)
  7 - 📁 Pastas temporárias (clipes_cortados, temp_raw)
  8 - 📦 Cache Python (módulos carregados)
  9 - 🔥 LIMPAR TUDO (exceto Drive)
 10 - 🔥🔥 LIMPAR TUDO + DRIVE (TUDO)
  0 - Sair
═════════════════════════════════════════════════════════════════

Digite sua escolha: 10

🎵 Removendo áudios...
   🗑️ pai_nosso_audio.wav

📝 Removendo legendas...
   🗑️ pai_nosso_es.srt
   🗑️ pai_nosso_es.es.srt
   🗑️ pai_nosso_fr.srt
   🗑️ pai_nosso_pt_edge.srt
   🗑️ 

### 🔵 Célula B0 — Opcional: baixar legendas do YouTube
Execute **antes** das fases 1–8 para ter traduções mais fiéis. Se pular, o Groq traduz a partir do PT.

In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CÉLULA B0 — Baixar legendas do YouTube                        ║
# ║  (BAIXA TAMBÉM A REFERÊNCIA PT para a FASE 3)                  ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import subprocess
import shutil
from drive_utils import DriveClient

cfg   = config
drive = DriveClient.get()

drive.download_se_ausente(cfg.ID_PASTA_COOKIES, cfg.NOME_COOKIES, Path(cfg.NOME_COOKIES))
cookies_flag = ['--cookies', cfg.NOME_COOKIES] if Path(cfg.NOME_COOKIES).exists() else []

print("═" * 60)
print("📺 BAIXANDO LEGENDAS DO YOUTUBE")
print("═" * 60)

# ── COLOQUE AS URLs DOS VÍDEOS COM LEGENDAS ABAIXO ────────────────────────────
URLS = {
    'pt': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'en': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'es': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
    'fr': 'https://www.youtube.com/watch?v=p5Vg7Vn2KeM',
}

for lang, url in URLS.items():
    nome_srt = cfg.nome_srt(lang)
    print(f'\n⬇️  Baixando {lang.upper()} de: {url[:50]}...')
    cmd = [
        'yt-dlp', '--write-sub', '--sub-lang', lang, '--write-auto-sub',
        '--skip-download', '--sub-format', 'srt', '--convert-subs', 'srt',
        '--output', f'{cfg.NOME_ORACAO}_{lang}', *cookies_flag, url
    ]
    resultado = subprocess.run(cmd, capture_output=True, text=True)
    encontrado = False
    for c in Path('.').glob(f'{cfg.NOME_ORACAO}_{lang}*.srt'):
        if c.name != nome_srt:
            c.rename(nome_srt)
        print(f'   ✅ {nome_srt} salvo')
        encontrado = True
        break
    if not encontrado:
        print(f'   ⚠️  Legenda {lang.upper()} não disponível nesse vídeo')

# Criar referência PT para a FASE 3
print("\n" + "═" * 60)
print("📝 PREPARANDO REFERÊNCIA PARA CORREÇÃO (FASE 3)")
print("═" * 60)
srt_pt   = Path(cfg.nome_srt('pt'))
ref_path = Path(f'{cfg.NOME_ORACAO}_yt_ref_pt.srt')
if srt_pt.exists():
    shutil.copy(srt_pt, ref_path)
    from srt_utils import ler_srt
    ref_legendas = ler_srt(ref_path)
    print(f'✅ Referência PT salva: {ref_path}  ({len(ref_legendas)} segmentos)')
    print(f'   Preview: {ref_legendas[0].texto[:80]}')
else:
    print(f'⚠️  SRT PT não encontrado. A FASE 3 corrigirá sem referência.')
print("═" * 60)

════════════════════════════════════════════════════════════
📺 BAIXANDO LEGENDAS DO YOUTUBE
════════════════════════════════════════════════════════════

⬇️  Baixando PT de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_pt.srt salvo

⬇️  Baixando EN de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_en.srt salvo

⬇️  Baixando ES de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_es.srt salvo

⬇️  Baixando FR de: https://www.youtube.com/watch?v=p5Vg7Vn2KeM...
   ✅ pai_nosso_fr.srt salvo

════════════════════════════════════════════════════════════
📝 PREPARANDO REFERÊNCIA PARA CORREÇÃO (FASE 3)
════════════════════════════════════════════════════════════
✅ Referência PT salva: pai_nosso_yt_ref_pt.srt  (11 segmentos)
   Preview: Pai nosso que estais no  céu,
════════════════════════════════════════════════════════════


### ▶ Fases 1–4 — Automáticas

In [5]:
# ── FASE 1: Áudio com Edge TTS ───────────────────────────────────────────────
audio = pipeline.fase1_gerar_audio()
print(f'✅ {audio}  ({audio.stat().st_size/1024:.0f} KB)')

# ── FASE 2: Transcrição Whisper ──────────────────────────────────────────────
srt_bruto = pipeline.fase2_transcrever_whisper()

from srt_utils import ler_srt
print(f'{srt_bruto.name}:')
for leg in ler_srt(srt_bruto)[:4]:
    print(f'  [{leg.inicio_str}]  {leg.texto}')

✅ pai_nosso_audio.wav  (162 KB)


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


pai_nosso_pt_edge.srt:
  [00:00:00,000]  Pai nosso que estáis no céu, santificados sejam o vosso nome.
  [00:00:05,540]  Vem a nós o vosso reino.
  [00:00:07,800]  Seja feita a vossa vontade, assim na Terra como no céu.
  [00:00:12,940]  O pão nosso de cada dia nos dá hoje.


In [9]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🗑️ REMOVER CACHE CORROMPIDO                                   ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path

# Remover cache corrompido da FASE 3
cache_pt = Path('/content/pai_nosso_pt_corrigido.json')
if cache_pt.exists():
    tamanho = cache_pt.stat().st_size
    if tamanho == 0:
        cache_pt.unlink()
        print("✅ Cache corrompido (vazio) removido!")
    else:
        print(f"⚠️ Cache tem {tamanho} bytes - mantido")

# Remover cache de traduções se existir e estiver corrompido
cache_traducoes = Path('/content/pai_nosso_traducoes.json')
if cache_traducoes.exists():
    tamanho = cache_traducoes.stat().st_size
    if tamanho == 0:
        cache_traducoes.unlink()
        print("✅ Cache de traduções corrompido removido!")

print("\n✅ Pronto! Agora execute a FASE 3")

✅ Cache corrompido (vazio) removido!

✅ Pronto! Agora execute a FASE 3


In [18]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🧪 TESTAR GEMINI API (chave nova)                              ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import userdata
from openai import OpenAI

GEMINI_KEY = userdata.get('GEMINI_API_KEY')

if GEMINI_KEY:
    client = OpenAI(
        api_key=GEMINI_KEY,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
    )

    try:
        resposta = client.chat.completions.create(
            model="gemini-2.0-flash",
            messages=[{"role": "user", "content": "Diga apenas: OK"}],
            max_tokens=10
        )
        print(f"✅ Gemini funcionou! Resposta: {resposta.choices[0].message.content}")
    except Exception as e:
        erro = str(e)
        if '429' in erro or 'rate limit' in erro.lower():
            print("❌ Gemini em RATE LIMIT! Aguarde alguns minutos.")
            print(f"   Detalhe: {erro[:200]}")
        else:
            print(f"❌ Erro: {erro}")
else:
    print("❌ Chave Gemini não encontrada no Secrets")

❌ Gemini em RATE LIMIT! Aguarde alguns minutos.
   Detalhe: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/g


In [17]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 3 — Correção PT com API Rotativa + DELAY                 ║
# ║  (Groq → Gemini → DeepSeek → Mistral)                          ║
# ║  Delay: 12 segundos entre legendas                             ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import time
import json
from datetime import datetime
from google.colab import userdata
from openai import OpenAI

print("═" * 70)
print("📝 FASE 3 — CORREÇÃO DO TEXTO EM PORTUGUÊS")
print("═" * 70)

# ═══════════════════════════════════════════════════════════════════
# CONFIGURAÇÕES
# ═══════════════════════════════════════════════════════════════════

DELAY_ENTRE_LEGENDAS = 12      # 12 segundos entre cada legenda

# ═══════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO DAS APIS (4 APIs!)
# ═══════════════════════════════════════════════════════════════════

APIS_CONFIG = [
    # 1. Groq
    {
        'nome': 'Groq',
        'secret_name': 'GROQ_KEY',
        'base_url': 'https://api.groq.com/openai/v1',
        'modelo': 'llama-3.3-70b-versatile',
    },
    # 2. Google Gemini
    {
        'nome': 'Gemini',
        'secret_name': 'GEMINI_API_KEY',
        'base_url': 'https://generativelanguage.googleapis.com/v1beta/openai/',
        'modelo': 'gemini-2.0-flash',
    },
    # 3. DeepSeek (mantido!)
    {
        'nome': 'DeepSeek',
        'secret_name': 'DEEP_SEEK_KEY',
        'base_url': 'https://api.deepseek.com/v1',
        'modelo': 'deepseek-chat',
    },
    # 4. Mistral
    {
        'nome': 'Mistral',
        'secret_name': 'MISTRAL_KEY',
        'base_url': 'https://api.mistral.ai/v1',
        'modelo': 'mistral-small-latest',
    },
]

class APIClienteRotativo:
    def __init__(self, configs):
        self.apis = []
        for cfg in configs:
            try:
                api_key = userdata.get(cfg['secret_name'])
                if api_key:
                    self.apis.append({
                        'nome': cfg['nome'],
                        'client': OpenAI(api_key=api_key, base_url=cfg['base_url']),
                        'modelo': cfg['modelo']
                    })
                    print(f"   ✅ {cfg['nome']} - carregada")
                else:
                    print(f"   ⚠️ {cfg['nome']} - chave não encontrada")
            except Exception as e:
                print(f"   ❌ {cfg['nome']} - erro: {e}")

        self.indice = 0
        self.estatisticas = {api['nome']: {'tentativas': 0, 'sucessos': 0} for api in self.apis}
        print(f"\n📊 Total: {len(self.apis)} API(s) disponíveis")

    def chamar(self, prompt, system_prompt="", max_tokens=500):
        for tentativa in range(len(self.apis) * 3):
            api = self.apis[self.indice]
            self.estatisticas[api['nome']]['tentativas'] += 1
            print(f"   🔄 Tentando: {api['nome']}...")

            try:
                messages = []
                if system_prompt:
                    messages.append({"role": "system", "content": system_prompt})
                messages.append({"role": "user", "content": prompt})

                resposta = api['client'].chat.completions.create(
                    model=api['modelo'],
                    messages=messages,
                    temperature=0.3,
                    max_tokens=max_tokens,
                )

                resultado = resposta.choices[0].message.content.strip()
                self.estatisticas[api['nome']]['sucessos'] += 1
                print(f"   ✅ {api['nome']} respondeu!")
                self.indice = 0
                return resultado

            except Exception as e:
                erro = str(e).lower()
                if 'rate limit' in erro or '429' in erro:
                    print(f"   ⚠️ {api['nome']} RATE LIMIT! Pulando...")
                elif '402' in erro or 'insufficient' in erro:
                    print(f"   ⚠️ {api['nome']} saldo insuficiente! Pulando...")
                elif 'invalid api key' in erro or 'authentication' in erro:
                    print(f"   ❌ {api['nome']} chave inválida! Pulando...")
                else:
                    print(f"   ⚠️ {api['nome']} erro: {erro[:50]}")

                self.indice = (self.indice + 1) % len(self.apis)
                time.sleep(3)
                continue

        return None

    def resumo(self):
        print("\n📊 ESTATÍSTICAS DAS APIS:")
        print("═" * 50)
        for nome, stats in self.estatisticas.items():
            if stats['tentativas'] > 0:
                taxa = stats['sucessos'] / stats['tentativas'] * 100
                print(f"   {nome}: {stats['sucessos']}/{stats['tentativas']} ({taxa:.0f}%)")
            else:
                print(f"   {nome}: nenhuma tentativa")
        print("═" * 50)

# ═══════════════════════════════════════════════════════════════════
# CARREGAR REFERÊNCIA PT
# ═══════════════════════════════════════════════════════════════════

ref_path = Path(f'{config.NOME_ORACAO}_yt_ref_pt.srt')
referencia = ""
if ref_path.exists():
    from srt_utils import ler_srt as _ler
    ref_legs = _ler(ref_path)
    referencia = " ".join([leg.texto for leg in ref_legs])
    print(f"✅ Referência PT: {len(ref_legs)} segmentos")
else:
    print(f"⚠️ Referência PT não encontrada")

# ═══════════════════════════════════════════════════════════════════
# CARREGAR LEGENDAS
# ═══════════════════════════════════════════════════════════════════

from srt_utils import ler_srt
srt_edge = Path(config.NOME_SRT_PT_EDGE)
legendas = ler_srt(srt_edge)
print(f"📊 {len(legendas)} legendas para corrigir")

print(f"\n⏱️  Delay entre legendas: {DELAY_ENTRE_LEGENDAS} segundos")
print("═" * 70)

# ═══════════════════════════════════════════════════════════════════
# CORRIGIR
# ═══════════════════════════════════════════════════════════════════

cliente = APIClienteRotativo(APIS_CONFIG)
inicio_total = datetime.now()

for i, leg in enumerate(legendas):
    print(f"\n{'─' * 50}")
    print(f"[{i+1}/{len(legendas)}] Original: {leg.texto[:60]}...")

    # Prompt mais direto
    if referencia:
        prompt = f"""Corrija APENAS este texto: "{leg.texto}"

Use como referência: {referencia[:300]}

Devolva SOMENTE o texto corrigido, sem explicações."""
    else:
        prompt = f"""Corrija APENAS este texto: "{leg.texto}"
Devolva SOMENTE o texto corrigido."""

    inicio_leg = datetime.now()
    corrigido = cliente.chamar(prompt, "Você é revisor de textos. Responda apenas com o texto corrigido.", max_tokens=500)
    tempo_leg = (datetime.now() - inicio_leg).seconds

    if corrigido and len(corrigido) < len(leg.texto) * 2:
        leg.texto = corrigido
        print(f"   ✅ Corrigido em {tempo_leg}s: {corrigido[:60]}...")
    else:
        print(f"   ❌ Falhou! Mantendo original.")

    if i < len(legendas) - 1:
        print(f"   ⏳ Aguardando {DELAY_ENTRE_LEGENDAS} segundos...")
        for s in range(DELAY_ENTRE_LEGENDAS, 0, -2):
            print(f"   ⏰ {s}s restantes...", end='\r')
            time.sleep(2)
        print()

# ═══════════════════════════════════════════════════════════════════
# SALVAR
# ═══════════════════════════════════════════════════════════════════

from srt_utils import eliminar_gaps, salvar_srt
legendas = eliminar_gaps(legendas)
srt_pt = Path(config.NOME_SRT_PT)
salvar_srt(legendas, srt_pt)

tempo_total = (datetime.now() - inicio_total).seconds
print("\n" + "═" * 70)
print(f"✅ FASE 3 CONCLUÍDA!")
print(f"   ⏱️  Tempo total: {tempo_total // 60}min {tempo_total % 60}s")
print(f"   📝 {len(legendas)} legendas corrigidas")
print("═" * 70)

print("\n📝 LEGENDAS CORRIGIDAS:")
print("─" * 50)
for leg in legendas:
    print(f'  #{leg.id:02d}  {leg.texto}')
print("─" * 50)

cliente.resumo()

══════════════════════════════════════════════════════════════════════
📝 FASE 3 — CORREÇÃO DO TEXTO EM PORTUGUÊS
══════════════════════════════════════════════════════════════════════
✅ Referência PT: 11 segmentos
📊 7 legendas para corrigir

⏱️  Delay entre legendas: 12 segundos
══════════════════════════════════════════════════════════════════════
   ✅ Groq - carregada
   ✅ Gemini - carregada
   ✅ DeepSeek - carregada
   ✅ Mistral - carregada

📊 Total: 4 API(s) disponíveis

──────────────────────────────────────────────────
[1/7] Original: Pai nosso que estáis no céu, santificados sejam o vosso nome...
   🔄 Tentando: Groq...
   ⚠️ Groq RATE LIMIT! Pulando...
   🔄 Tentando: Gemini...
   ⚠️ Gemini RATE LIMIT! Pulando...
   🔄 Tentando: DeepSeek...
   ⚠️ DeepSeek saldo insuficiente! Pulando...
   🔄 Tentando: Mistral...
   ✅ Mistral respondeu!
   ✅ Corrigido em 11s: Pai nosso que estais no céu, santificado seja o vosso nome....
   ⏳ Aguardando 12 segundos...


─────────────────────────────

### ▶ Fase 5A — Classificação morfológica (Groq) + Pacote de revisão

O pipeline vai:
1. Classificar todos os idiomas via Groq (forçando do zero)
2. Salvar os JSONs brutos em `brutos/` no Drive
3. **Gerar o pacote de revisão** completo em `correcoes/` no Drive:
   - `prompt_revisao.md` — instruções para a IA revisora
   - `relatorio_classificacoes.csv` — todas as palavras com coluna `Sugestao`
   - Os 4 JSONs brutos prontos para correção
4. **Pausar** e mostrar instruções para a Fase 5B

In [20]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 5A — Classificação Morfológica (usando SRT CORRIGIDO)    ║
# ║  (Groq → Mistral) | Delay: 20s entre legendas                  ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path
import time
import json
import re
from datetime import datetime
from google.colab import userdata
from openai import OpenAI
from srt_utils import ler_srt, salvar_srt
from models import Palavra, Legenda

print("═" * 70)
print("🤖 FASE 5A — CLASSIFICAÇÃO MORFOLÓGICA")
print("═" * 70)

# ═══════════════════════════════════════════════════════════════════
# CONFIGURAÇÕES
# ═══════════════════════════════════════════════════════════════════

DELAY_ENTRE_LEGENDAS = 20      # 20 segundos entre cada legenda
DELAY_ENTRE_IDIOMAS = 40       # 40 segundos entre idiomas
DELAY_RATE_LIMIT = 90          # 90 segundos se bater rate limit

# Pastas de cache
CACHE_DIR = Path('/content/cache_classificacao')
CACHE_DIR.mkdir(exist_ok=True)

# ═══════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO DAS APIS (Groq → Mistral)
# ═══════════════════════════════════════════════════════════════════

APIS_CONFIG = [
    {
        'nome': 'Groq',
        'secret_name': 'GROQ_KEY',
        'base_url': 'https://api.groq.com/openai/v1',
        'modelo': 'llama-3.3-70b-versatile',
    },
    {
        'nome': 'Mistral',
        'secret_name': 'MISTRAL_KEY',
        'base_url': 'https://api.mistral.ai/v1',
        'modelo': 'mistral-small-latest',
    },
]

def limpar_json(texto):
    """Tenta extrair e corrigir JSON mal formatado"""
    if not texto:
        return None

    # Tentar encontrar JSON entre chaves
    match = re.search(r'\{[^{}]*"palavras"\s*:\s*\[.*?\]\s*\}', texto, re.DOTALL)
    if not match:
        match = re.search(r'\{.*\}', texto, re.DOTALL)

    if not match:
        return None

    json_str = match.group()

    # Corrigir aspas simples para duplas
    json_str = re.sub(r"'", '"', json_str)

    # Remover trailing commas
    json_str = re.sub(r',\s*}', '}', json_str)
    json_str = re.sub(r',\s*]', ']', json_str)

    # Remover caracteres de controle
    json_str = re.sub(r'[\x00-\x1f\x7f-\x9f]', '', json_str)

    try:
        return json.loads(json_str)
    except:
        # Última tentativa: encontrar padrão de palavras
        palavras = re.findall(r'"texto"\s*:\s*"([^"]+)"\s*,\s*"classe"\s*:\s*"([^"]+)"', json_str)
        if palavras:
            return {"palavras": [{"texto": p[0], "classe": p[1]} for p in palavras]}

    return None

class APIClienteRotativo:
    def __init__(self, configs):
        self.apis = []
        for cfg in configs:
            try:
                api_key = userdata.get(cfg['secret_name'])
                if api_key:
                    self.apis.append({
                        'nome': cfg['nome'],
                        'client': OpenAI(api_key=api_key, base_url=cfg['base_url']),
                        'modelo': cfg['modelo']
                    })
                    print(f"   ✅ {cfg['nome']} - carregada")
                else:
                    print(f"   ⚠️ {cfg['nome']} - chave não encontrada")
            except Exception as e:
                print(f"   ❌ {cfg['nome']} - erro: {e}")

        self.indice = 0
        self.estatisticas = {api['nome']: {'tentativas': 0, 'sucessos': 0} for api in self.apis}
        print(f"\n📊 Total: {len(self.apis)} API(s) disponíveis")

    def classificar_legenda(self, texto, lang):
        """Classifica palavras de uma legenda usando JSON limpo"""
        prompt = f"""Idioma: {lang}
Texto: "{texto}"

Classifique cada palavra gramaticalmente. Use classes específicas como:
- substantivo_masculino_singular, substantivo_feminino_singular
- pronome_relativo, pronome_possessivo_singular, pronome_pessoal
- verbo_presente, verbo_passado, verbo_imperativo
- adjetivo_normal, advérbio_normal, preposicao, artigo_definido, conjuncao

REGRAS:
- 'que' em PT/ES/FR é pronome_relativo
- verbos no infinitivo são verbo_presente

Responda APENAS com JSON válido. Exemplo:
{{"palavras": [{{"texto": "Pai", "classe": "substantivo_masculino_singular"}}]}}"""

        for tentativa in range(len(self.apis) * 3):
            api = self.apis[self.indice]
            self.estatisticas[api['nome']]['tentativas'] += 1
            print(f"   🔄 Tentando: {api['nome']}...")

            try:
                messages = [
                    {"role": "system", "content": "Você é especialista em linguística. Responda apenas com JSON válido e bem formatado."},
                    {"role": "user", "content": prompt}
                ]

                resposta = api['client'].chat.completions.create(
                    model=api['modelo'],
                    messages=messages,
                    temperature=0.2,
                    max_tokens=800,
                )

                resultado = resposta.choices[0].message.content.strip()

                # Limpar e parsear JSON
                dados = limpar_json(resultado)
                if dados and 'palavras' in dados:
                    self.estatisticas[api['nome']]['sucessos'] += 1
                    print(f"   ✅ {api['nome']} respondeu!")
                    self.indice = 0

                    palavras = []
                    for p in dados['palavras']:
                        if p.get('texto'):
                            palavras.append(Palavra(
                                texto=p['texto'],
                                classe=p.get('classe', 'substantivo_masculino_singular')
                            ))
                    return palavras
                else:
                    print(f"   ⚠️ {api['nome']} JSON inválido, tentando novamente...")

            except Exception as e:
                erro = str(e).lower()
                if 'rate limit' in erro or '429' in erro:
                    print(f"   ⚠️ {api['nome']} RATE LIMIT! Aguardando {DELAY_RATE_LIMIT}s...")
                    time.sleep(DELAY_RATE_LIMIT)
                else:
                    print(f"   ⚠️ {api['nome']} erro: {erro[:50]}")

            self.indice = (self.indice + 1) % len(self.apis)
            time.sleep(5)

        print(f"   ❌ Todas as APIs falharam!")
        return [Palavra(texto=t, classe='substantivo_masculino_singular') for t in texto.split()]

    def resumo(self):
        print("\n📊 ESTATÍSTICAS DAS APIS:")
        print("═" * 50)
        for nome, stats in self.estatisticas.items():
            if stats['tentativas'] > 0:
                taxa = stats['sucessos'] / stats['tentativas'] * 100
                print(f"   {nome}: {stats['sucessos']}/{stats['tentativas']} ({taxa:.0f}%)")
            else:
                print(f"   {nome}: nenhuma tentativa")
        print("═" * 50)

# ═══════════════════════════════════════════════════════════════════
# FUNÇÕES DE CACHE
# ═══════════════════════════════════════════════════════════════════

def get_cache_path(lang, legenda_id):
    return CACHE_DIR / f'{config.NOME_ORACAO}_{lang}_leg{legenda_id}.json'

def carregar_do_cache(lang, legendas):
    cache_carregado = 0
    for leg in legendas:
        cache_path = get_cache_path(lang, leg.id)
        if cache_path.exists():
            try:
                with open(cache_path, 'r', encoding='utf-8') as f:
                    dados = json.load(f)
                leg.palavras = [Palavra(texto=p['texto'], classe=p['classe']) for p in dados]
                cache_carregado += 1
            except:
                pass
    return cache_carregado

def salvar_no_cache(lang, legenda):
    cache_path = get_cache_path(lang, legenda.id)
    dados = [{'texto': p.texto, 'classe': p.classe} for p in legenda.palavras]
    with open(cache_path, 'w', encoding='utf-8') as f:
        json.dump(dados, f, indent=2, ensure_ascii=False)

# ═══════════════════════════════════════════════════════════════════
# CARREGAR LEGENDAS (USANDO O SRT CORRIGIDO!)
# ═══════════════════════════════════════════════════════════════════

# ✅ CORRIGIDO: usar o SRT corrigido, não o bruto!
srt_corrigido = Path(config.NOME_SRT_PT)
if not srt_corrigido.exists():
    print(f"⚠️ SRT corrigido não encontrado: {srt_corrigido}")
    print("   Certifique-se de que a FASE 3 foi executada primeiro!")
else:
    legendas_pt = ler_srt(srt_corrigido)
    print(f"📖 Legendas PT carregadas do SRT CORRIGIDO: {len(legendas_pt)} segmentos")

    # Mostrar preview para confirmar
    print("\n📝 Preview das legendas CORRIGIDAS:")
    for leg in legendas_pt[:3]:
        print(f"   #{leg.id:02d} {leg.texto[:60]}...")

# Carregar traduções
if not pipeline.legendas_idiomas:
    pipeline.legendas_idiomas = pipeline._carregar_todos_srts(legendas_pt)

print(f"\n📊 Total de legendas para classificar: {sum(len(pipeline.legendas_idiomas[lang]) for lang in config.IDIOMAS)}")
print(f"⏱️  Delay entre legendas: {DELAY_ENTRE_LEGENDAS}s")
print(f"⏱️  Delay entre idiomas: {DELAY_ENTRE_IDIOMAS}s")
print("═" * 70)

# ═══════════════════════════════════════════════════════════════════
# CLASSIFICAR (COM CACHE)
# ═══════════════════════════════════════════════════════════════════

cliente = APIClienteRotativo(APIS_CONFIG)
inicio_total = datetime.now()

for i, lang in enumerate(config.IDIOMAS):
    print(f"\n{'='*60}")
    print(f"🔤 {lang.upper()} - {len(pipeline.legendas_idiomas[lang])} legendas")
    print(f"{'='*60}")

    # Carregar do cache primeiro
    cache_carregado = carregar_do_cache(lang, pipeline.legendas_idiomas[lang])
    if cache_carregado > 0:
        print(f"📁 Cache: {cache_carregado}/{len(pipeline.legendas_idiomas[lang])} legendas carregadas")

    for j, leg in enumerate(pipeline.legendas_idiomas[lang]):
        # Pular se já está no cache
        if get_cache_path(lang, leg.id).exists() and leg.palavras:
            print(f"   [{j+1}/{len(pipeline.legendas_idiomas[lang])}] ⏭️ Pulando (cache) - {leg.texto[:40]}...")
            continue

        print(f"\n   📝 [{j+1}/{len(pipeline.legendas_idiomas[lang])}] {leg.texto[:50]}...")

        inicio_leg = datetime.now()
        palavras = cliente.classificar_legenda(leg.texto, lang)
        tempo_leg = (datetime.now() - inicio_leg).seconds

        if palavras:
            leg.palavras = palavras
            salvar_no_cache(lang, leg)
            print(f"   ✅ Classificada em {tempo_leg}s ({len(palavras)} palavras)")
        else:
            print(f"   ❌ Falhou!")

        # DELAY entre legendas
        if j < len(pipeline.legendas_idiomas[lang]) - 1:
            print(f"   ⏳ Aguardando {DELAY_ENTRE_LEGENDAS}s...")
            for s in range(DELAY_ENTRE_LEGENDAS, 0, -5):
                print(f"   ⏰ {s}s restantes...", end='\r')
                time.sleep(5)
            print()

    # DELAY entre idiomas
    if i < len(config.IDIOMAS) - 1:
        print(f"\n⏳ Aguardando {DELAY_ENTRE_IDIOMAS}s antes do próximo idioma...")
        for s in range(DELAY_ENTRE_IDIOMAS, 0, -10):
            print(f"   ⏰ {s}s restantes...", end='\r')
            time.sleep(10)
        print()

# ═══════════════════════════════════════════════════════════════════
# SALVAR JSONs
# ═══════════════════════════════════════════════════════════════════

print("\n" + "═" * 70)
print("📦 SALVANDO CLASSIFICAÇÕES...")
print("═" * 70)

for lang in config.IDIOMAS:
    dados = {}
    for leg in pipeline.legendas_idiomas[lang]:
        dados[str(leg.id)] = {
            'inicio': leg.inicio_str,
            'fim': leg.fim_str,
            'texto_original': leg.texto,
            'palavras': [{'texto': p.texto, 'classe': p.classe} for p in leg.palavras]
        }

    json_path = Path(f'classificacao_{config.NOME_ORACAO}_{lang}.json')
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(dados, f, indent=2, ensure_ascii=False)
    print(f"   ✅ classificacao_{config.NOME_ORACAO}_{lang}.json")

tempo_total = (datetime.now() - inicio_total).seconds
print("\n" + "═" * 70)
print(f"✅ CLASSIFICAÇÃO CONCLUÍDA!")
print(f"   ⏱️  Tempo total: {tempo_total // 60}min {tempo_total % 60}s")
print("═" * 70)

cliente.resumo()

print("\n📦 Para gerar o pacote de revisão, execute:")
print("   pacote = pipeline._clf.exportar_pacote_revisao(pipeline.legendas_idiomas)")

══════════════════════════════════════════════════════════════════════
🤖 FASE 5A — CLASSIFICAÇÃO MORFOLÓGICA
══════════════════════════════════════════════════════════════════════
📖 Legendas PT carregadas do SRT CORRIGIDO: 7 segmentos

📝 Preview das legendas CORRIGIDAS:
   #01 Pai nosso que estais no céu, santificado seja o vosso nome....
   #02 Venha a nós o vosso reino....
   #03 Seja feita a vossa vontade, assim na terra como no céu....

📊 Total de legendas para classificar: 28
⏱️  Delay entre legendas: 20s
⏱️  Delay entre idiomas: 40s
══════════════════════════════════════════════════════════════════════
   ✅ Groq - carregada
   ✅ Mistral - carregada

📊 Total: 2 API(s) disponíveis

🔤 PT - 7 legendas
📁 Cache: 7/7 legendas carregadas
   [1/7] ⏭️ Pulando (cache) - Pai nosso que estais no céu, santificado...
   [2/7] ⏭️ Pulando (cache) - Venha a nós o vosso reino....
   [3/7] ⏭️ Pulando (cache) - Pai nosso que estais no céu, santificado...
   [4/7] ⏭️ Pulando (cache) - O pão nosso de c

KeyboardInterrupt: 

### ⏸️ Fase 5B — Revisão manual pela IA ← **Você faz isso**

Após a Fase 5A concluir:

1. **Abra o Google Drive** → `MyDrive/pipeline/correcoes/pai_nosso/`
2. Você verá:
   - `prompt_revisao.md` ← **copie e cole numa IA junto com os 4 JSONs**
   - `relatorio_classificacoes.csv` ← consulte a coluna `Sugestao` para ver correções sugeridas
   - Os 4 JSONs brutos para corrigir
3. A IA retorna os JSONs corrigidos
4. **Salve os JSONs corrigidos na mesma pasta** (substituindo os originais)
5. Execute a Fase 5C abaixo

---
### ▶ Fase 5C — Recarregar classificações corrigidas

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  FASE 5C — Recarregar do Drive após revisão                    ║
# ║  Execute APÓS salvar os JSONs corrigidos no Drive              ║
# ╚══════════════════════════════════════════════════════════════════╝

from pathlib import Path

print("═" * 65)
print("📥 FASE 5C — RECARREGANDO CLASSIFICAÇÕES CORRIGIDAS")
print("═" * 65)

pasta_correcoes = Path(f'/content/drive/MyDrive/pipeline/correcoes/{config.NOME_ORACAO}')

if not pasta_correcoes.exists():
    print(f"❌ Pasta não encontrada: {pasta_correcoes}")
    print("   Execute a FASE 5A primeiro.")
else:
    jsons_ok = []
    jsons_faltando = []
    for lang in config.IDIOMAS:
        p = pasta_correcoes / f'classificacao_{config.NOME_ORACAO}_{lang}.json'
        if p.exists():
            jsons_ok.append(lang.upper())
            print(f"   ✅ {p.name}")
        else:
            jsons_faltando.append(lang.upper())
            print(f"   ❌ {p.name}")

    if len(jsons_ok) == 4:
        print("\n✅ Todos os 4 JSONs encontrados! Carregando...")
        pipeline.legendas_idiomas = pipeline.fase5b_recarregar()

        print("\n📋 Preview das classificações:")
        print("─" * 55)
        for lang in config.IDIOMAS:
            legs = pipeline.legendas_idiomas.get(lang, [])
            if legs and legs[0].palavras:
                leg = legs[0]
                print(f'\n  {lang.upper()} — "{leg.texto[:50]}"')
                from constants import CORES_HTML
                for p in leg.palavras[:4]:
                    ok  = '✅' if p.classe in CORES_HTML else '❌'
                    print(f'    {ok} {p.texto:<18} {p.classe}')
                if len(leg.palavras) > 4:
                    print(f'       ... (+{len(leg.palavras)-4} palavras)')

        print("\n" + "═" * 65)
        print("🎬 Pronto — execute as Fases 6, 7 e 8")
        print("═" * 65)
    else:
        print(f"\n⚠️  Faltam: {', '.join(jsons_faltando)}")
        print("   Coloque os JSONs corrigidos na pasta e execute novamente.")

### ▶ Fases 6–8 — Vídeo final (rode após a Fase 5C)

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  FASES 6, 7 e 8 — Vídeo final                 ║
# ╚══════════════════════════════════════════════════╝

video_final = pipeline.continuar()

if video_final:
    print(f'\n🎬 Vídeo final: {video_final}')
    print(f'   Tamanho: {video_final.stat().st_size / 1_048_576:.1f} MB')
    print(f'   Salvo no Drive: pasta Vídeos')

### 🚀 RUN ALL — Pipeline completo com checkpoint

Use quando quiser rodar tudo de uma vez. O pipeline pausa automaticamente na Fase 5A
se houver classificações a revisar. Após a revisão, rode a célula da Fase 5C
e depois a célula das Fases 6–8.

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  RUN ALL                                       ║
# ╚══════════════════════════════════════════════════╝
from video_pipeline import ClassificacaoPendenteError

resultado = pipeline.run(
    # from_phase='clipes_cortados'  # descomente para retomar de uma fase
)

if resultado is None:
    print('\n⏸️  Pipeline pausado na Fase 5A.')
    print('   Siga as instruções acima, depois execute a célula FASE 5C.')
else:
    print(f'\n🎉 Vídeo final: {resultado}')
    print(pipeline._cp.resumo())

### 🔧 Utilitários

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  UTILITÁRIOS — descomente o que precisar       ║
# ╚══════════════════════════════════════════════════╝
from checkpoint import Checkpoint
cp = Checkpoint()
print(cp.resumo())
print()
pipeline._clf.imprimir_status()

# ── Checkpoint ────────────────────────────────────────────────────────────────
# cp.resetar_tudo()
# cp.reiniciar_de('classificacoes_feitas')

# ── Classificações ────────────────────────────────────────────────────────────
# pipeline._clf.imprimir_status()
# pipeline._clf.classificar_idioma(pipeline.legendas_idiomas['en'], 'en', forcar=True)

# Gerar pacote de revisão manualmente (sem reclassificar):
# pacote = pipeline._clf.exportar_pacote_revisao(pipeline.legendas_idiomas)
# print(f"📦 Pacote: {pacote}")

# Verificar classes inválidas:
# from constants import CORES_HTML
# for lang, legendas in pipeline.legendas_idiomas.items():
#     for leg in legendas:
#         for p in leg.palavras:
#             if p.classe not in CORES_HTML:
#                 print(f'  [{lang.upper()} leg.{leg.id}] "{p.texto}" → {p.classe} ❌')